# Preprocessing Data for Finetuning NEMO

- Creator: Arpit Merchant (am84@sanger.ac.uk)
- Date of Creation: 21.10.2025
- Date of Last Modification: 22.10.2025 (by Arpit Merchant; am84@sanger.ac.uk)

## Load Tokenized Corpus

In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
from datasets import load_from_disk
import numpy as np
import pickle
import re
import os

In [14]:
NUM_CORES = int(os.environ.get("LSB_DJOB_NUMPROC",1))
print(NUM_CORES)

4


In [3]:
data_path = "/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset"
dataset = load_from_disk(data_path)
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [4]:
for k, v in dataset[0].items():
    if k in ['gene_tokens', 'gene_expr']:
        print(f"{k} (length): {len(v)}")
    else:
        print(f"{k}: {v}")
    print("")

rel_x_coord: [0.0, -7.36981201171875, 3.20379638671875, 8.82733154296875, 1.98760986328125, -8.05316162109375, -7.29248046875, 6.572998046875, -14.7947998046875, -2.12060546875, 15.0740966796875]

rel_y_coord: [0.0, 1.64208984375, -7.16650390625, 1.13330078125, 9.93310546875, -8.265625, 11.01611328125, 15.4580078125, -8.0322265625, 17.55078125, 9.50732421875]

cell_id: 23_batch0_0

batch_value: [166]

gene_panel_value: [127]

assay_value: [122]

species_value: [102]

tissue_value: [103]

gene_tokens (length): 2816

gene_expr (length): 2816

n_nonzero_tokens: 910



In [6]:
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 88626282
})

## Filtering Methods

In [21]:
def filter_dataset_by_dataset_batch(
    dataset,
    target_dataset_batch_ids,
    batch_size=10000,
    num_proc=4,
):
    """
    Filter a Hugging Face dataset to keep only cells matching a given dataset ID and batch IDs.
    
    Parameters:
    -----------
    dataset : Dataset
        Hugging Face dataset with 'cell_id' column
    target_dataset_batch_ids : list
        List of dataset id and batch id to filter for
    batch_size : int, default=10000
        Batch size for processing
    num_proc : int, default=4
        Number of processes for parallel processing
        
    Returns:
    --------
    Dataset
        Filtered dataset containing only matching items
    """
    # Define the filter function
    def filter_function(ids):
        return [i.split('_')[0] + '_' + i.split('_')[1] in target_dataset_batch_ids for i in ids]
    
    # Apply the filter
    filtered_dataset = dataset.filter(
        filter_function,
        input_columns="cell_id",
        batched=True,
        batch_size=batch_size,
        num_proc=num_proc,
    )
    
    return filtered_dataset

## Filter Toy Dataset: Section 1 from xhs1000-39b_1p

In [18]:
toy_train_dataset_filter = filter_dataset_by_dataset_batch(
    dataset, 
    target_dataset_batch_ids=['1000_batch1'],
    batch_size=10000,
    num_proc=NUM_CORES
)

Filtering by dataset_id and batch_id (num_proc=4): 100%|██████████| 88626282/88626282 [09:05<00:00, 162398.34 examples/s]


In [19]:
toy_train_dataset_filter

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 8532
})

In [ ]:
toy_train_dataset_filter.save_to_disk("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/toy_train_dataset_xhs1000-39b_1p-batch1.dataset")